<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px 30px; border-radius: 12px; margin-bottom: 10px;">
    <h1 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:2.2em; margin:0 0 8px 0;">
        🎓 Introdução ao Aprendizado de Máquina
    </h1>
    <h2 style="color:#a8d8ea; font-family:'Segoe UI', sans-serif; font-size:1.3em; margin:0 0 6px 0; font-weight:400;">
        Aula 04 — Regressão Linear
    </h2>
    <h3 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:1.05em; margin:0 0 12px 0; font-weight:500;">
        🔬 Prática — Prevendo o Preço das Passagens do Titanic
    </h3>
    <p style="color:#ccc; font-family:'Segoe UI', sans-serif; font-size:0.9em; margin:0;">
        Prof. Felipe Amaral
    </p>
</div>
<div style="display:flex; gap:10px; margin-top:10px; flex-wrap:wrap;">
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">📚 FIAP</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🐍 Python 3</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🚢 Dataset Titanic</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">📈 Regressão</span>
</div>


## O primeiro modelo do curso

Nas Aulas 02 e 03 você explorou, limpou e preparou os dados do Titanic — mas ainda não treinou
nenhum modelo. Hoje isso muda: vamos treinar o **primeiro modelo de Machine Learning do curso**,
a Regressão Linear.

Até agora, quando pensamos em "prever algo", geralmente imaginamos uma categoria (sim/não,
tipo A/B/C). Hoje vamos prever **um número**. Essa tarefa se chama **regressão**.

```
CLASSIFICAÇÃO (mais adiante no curso)     REGRESSÃO (hoje)
──────────────────────────────────       ─────────────────────────────────
Pergunta: "Esse passageiro sobreviveu?"  Pergunta: "Quanto custou a passagem?"
Resposta: SIM ou NÃO                     Resposta: £ 7.25  ou  £ 156.50
```

### O problema de hoje

O preço das passagens do Titanic variava muito — de poucas libras na 3ª classe até
centenas de libras na 1ª classe. Vamos construir um modelo que, olhando para as
características de um passageiro (classe, gênero, tamanho da família), **estima quanto
ele pagou pela passagem (`fare`)**.

Esse tipo de problema aparece o tempo todo no mundo real: estimar o preço de um
imóvel, prever o valor de uma venda, calcular um seguro. A ideia é sempre a mesma —
usar características conhecidas (X) para estimar um número desconhecido (Y).

### Roteiro de hoje

| Parte | Tema |
|-------|------|
| **1** | Conhecendo o alvo: a tarifa (`fare`) |
| **2** | A reta da Regressão Linear — o que ela aprende |
| **3** | Regressão Múltipla no Titanic |
| **4** | Avaliando o modelo: métricas e resíduos |

> **Tempo estimado: 30 minutos**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize":  (10, 5),
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.titlesize":     13,
    "axes.labelsize":     11,
})
sns.set_theme(style="whitegrid", palette="muted")

# ── Carregando e preparando o Titanic (mesma limpeza das aulas anteriores) ────
df = sns.load_dataset("titanic").copy()

df["age"]      = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.drop(columns=["deck"]).drop_duplicates().reset_index(drop=True)

df["tamanho_familia"] = df["sibsp"] + df["parch"] + 1
df["sex_enc"]         = (df["sex"] == "female").astype(int)

# Removendo os poucos registros com fare = 0 (dado inconsistente)
df = df[df["fare"] > 0].reset_index(drop=True)

print("✅ Dataset pronto!")
print(f"   {len(df)} passageiros com tarifa > 0")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 1</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Conhecendo o Alvo: a Tarifa (fare)</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Antes de prever um número, é preciso entender como ele se distribui."</p>
    </div>
</div>

Nosso objetivo (a variável **Y**, ou "alvo") é `fare` — quanto cada passageiro pagou
pela passagem, em libras (£).

Antes de treinar qualquer modelo, vale a pena **olhar para os dados**. O gráfico
abaixo mostra como as tarifas se distribuem, e como elas variam por classe.


In [ ]:
# Visualizando o nosso alvo: distribuição das tarifas
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Variável Alvo — Fare (Tarifa da Passagem) £", fontweight="bold")

# Histograma
axes[0].hist(df["fare"], bins=40, color="#0f3460", edgecolor="white", alpha=0.85)
axes[0].axvline(df["fare"].mean(),   color="#e94560", linestyle="--",
                linewidth=2, label=f"Média: £{df['fare'].mean():.0f}")
axes[0].axvline(df["fare"].median(), color="#f0a500", linestyle="--",
                linewidth=2, label=f"Mediana: £{df['fare'].median():.0f}")
axes[0].set_xlabel("Tarifa (£)"); axes[0].set_ylabel("Nº de passageiros")
axes[0].set_title("Distribuição das Tarifas")
axes[0].legend()

# Boxplot por classe
df.boxplot(column="fare", by="pclass", ax=axes[1],
           boxprops=dict(color="#0f3460"),
           medianprops=dict(color="#e94560", linewidth=2.5),
           whiskerprops=dict(color="#0f3460"),
           capprops=dict(color="#0f3460"))
axes[1].set_xlabel("Classe"); axes[1].set_ylabel("Tarifa (£)")
axes[1].set_title("Tarifa por Classe")
plt.suptitle("")

plt.tight_layout()
plt.show()

print(f"A passagem mais barata custou: £{df['fare'].min():.2f}")
print(f"A mais cara custou:            £{df['fare'].max():.2f}")
print(f"Média:                         £{df['fare'].mean():.2f}")
print(f"Mediana:                       £{df['fare'].median():.2f}")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 1 — Olhando para os dois gráficos acima, responda: (a) por que a média das tarifas é maior que a mediana? (b) a 1ª classe custa visivelmente mais que a 3ª — você diria que existe uma relação clara entre classe e tarifa?</span></div>

*✏️ (a) A média é maior que a mediana porque: `???`*

*✏️ (b) Existe relação entre classe e tarifa? `???`*


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 2</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">A Reta da Regressão Linear</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Um modelo de regressão simples é, no fundo, uma reta."</p>
    </div>
</div>

### A ideia central

A Regressão Linear Simples tenta desenhar a **melhor reta possível** que passa
pelos pontos dos dados. Essa reta é descrita por uma equação bem simples:

```
ŷ = β₀ + β₁ · x

  x  = a característica que conhecemos (ex: classe do passageiro)
  ŷ  = o valor que o modelo prevê ("y chapéu")
  β₀ = intercepto  → valor de ŷ quando x = 0
  β₁ = inclinação  → quanto ŷ muda para cada +1 unidade de x
```

Pense assim: **β₁ é a "taxa de câmbio"** entre a feature e o alvo. Se β₁ = -20,
cada classe a mais (de 1ª para 2ª, por exemplo) reduz a tarifa prevista em £20.

### Como o computador encontra essa reta?

Existem **infinitas retas possíveis** passando perto desses pontos. Como o
computador escolhe uma só? Ele testa: para cada reta candidata, mede o quanto
ela erra em cada ponto, eleva esse erro ao quadrado (para que erro positivo e
erro negativo não se cancelem) e soma tudo. Essa soma é o "placar de erro" da
reta — quanto **menor**, melhor a reta.

```
Para cada reta candidata:

  erro em cada ponto  =  valor real  −  valor previsto pela reta
  "placar de erro"    =  soma de (erro)² de todos os pontos
```

A reta escolhida é simplesmente a que tem o **menor placar de erro** dentre
todas as possíveis. Esse método se chama **Mínimos Quadrados (OLS)**. Vamos
ver isso concretamente: comparando a reta "certa" com uma reta "chutada" e
comparando o placar de erro das duas.


In [ ]:
# Comparando duas retas candidatas: qual tem o menor "placar de erro"?
x_demo = np.array([1, 2, 3])   # classe do passageiro
y_demo = np.array([df[df.pclass == c]["fare"].mean() for c in [1, 2, 3]])

# Reta A: a melhor reta possível, encontrada pelo método dos mínimos quadrados
beta1_A, beta0_A = np.polyfit(x_demo, y_demo, 1)

# Reta B: uma reta "chutada" (inclinação mais suave, só para comparar)
beta1_B, beta0_B = -15.0, 70.0

def placar_de_erro(beta0, beta1, x, y):
    y_previsto = beta0 + beta1 * x
    erros = y - y_previsto
    return (erros ** 2).sum()   # soma dos erros ao quadrado

placar_A = placar_de_erro(beta0_A, beta1_A, x_demo, y_demo)
placar_B = placar_de_erro(beta0_B, beta1_B, x_demo, y_demo)

print("Reta A (mínimos quadrados): ŷ = "
      f"{beta0_A:.1f} + ({beta1_A:.1f})·x   →  placar de erro = {placar_A:,.0f}")
print("Reta B (chutada):           ŷ = "
      f"{beta0_B:.1f} + ({beta1_B:.1f})·x   →  placar de erro = {placar_B:,.0f}")
print()
print("A reta com o MENOR placar de erro é a que o computador escolhe.")


In [ ]:
# Visualizando as duas retas lado a lado
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x_line = np.linspace(0.5, 3.5, 100)

for ax, beta0, beta1, placar, titulo in [
    (axes[0], beta0_A, beta1_A, placar_A, "Reta A — Mínimos Quadrados"),
    (axes[1], beta0_B, beta1_B, placar_B, "Reta B — Chutada"),
]:
    y_previsto = beta0 + beta1 * x_demo
    ax.plot(x_line, beta0 + beta1 * x_line, color="#0f3460", linewidth=2.5)
    ax.scatter(x_demo, y_demo, color="#e94560", s=150, zorder=5,
               label="Tarifa média real", edgecolors="white", linewidth=1.5)
    for xi, yi, yi_hat in zip(x_demo, y_demo, y_previsto):
        ax.vlines(xi, yi_hat, yi, color="#f0a500", linewidth=2, linestyle="--")
    ax.set_xlabel("Classe"); ax.set_ylabel("Tarifa média (£)")
    ax.set_title(f"{titulo}\nplacar de erro = {placar:,.0f}", fontweight="bold")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("Repare: na Reta B, as linhas tracejadas (erros) são bem maiores —")
print("por isso o placar de erro dela é maior, e o computador prefere a Reta A.")


Esse foi só um exemplo com 2 retas para você **ver** a ideia funcionando.
Na prática, o computador não testa retas uma por uma — ele usa uma fórmula
matemática que já calcula direto a reta de menor placar de erro. É exatamente
isso que a função `np.polyfit` faz por baixo dos panos. Vamos usá-la agora
para o nosso problema real: prever a tarifa média a partir da classe.


In [ ]:
# Regressão simples: classe (x) prevendo a tarifa média (y)
x = np.array([1, 2, 3])   # classe do passageiro
y = np.array([df[df.pclass == c]["fare"].mean() for c in [1, 2, 3]])

# np.polyfit encontra a melhor reta (grau 1) automaticamente
beta1, beta0 = np.polyfit(x, y, 1)

print("Tarifa média por classe:")
for c, tarifa in zip(x, y):
    print(f"  Classe {c}: £{tarifa:.2f}")

print(f"\nReta encontrada:  ŷ = {beta0:.2f} + ({beta1:.2f}) · x")
print(f"\nInterpretação: a cada classe a mais (1ª → 2ª → 3ª),")
print(f"a tarifa prevista muda em £{beta1:.2f}.")


In [ ]:
# Visualizando a reta sobre os pontos reais
y_hat = beta0 + beta1 * x
x_line = np.linspace(0.5, 3.5, 100)
y_line = beta0 + beta1 * x_line

plt.figure(figsize=(8, 5))
plt.plot(x_line, y_line, color="#0f3460", linewidth=2.5,
         label=f"Reta: ŷ = {beta0:.1f} + ({beta1:.1f})·x")
plt.scatter(x, y, color="#e94560", s=150, zorder=5,
            label="Tarifa média real", edgecolors="white", linewidth=1.5)

for xi, yi, yi_hat in zip(x, y, y_hat):
    plt.vlines(xi, yi_hat, yi, color="#f0a500", linewidth=2, linestyle="--")

plt.xlabel("Classe"); plt.ylabel("Tarifa média (£)")
plt.title("A reta da Regressão Linear — Classe vs Tarifa Média", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()

print("As linhas tracejadas amarelas são os 'erros' (resíduos) da reta —")
print("a diferença entre o valor real e o valor previsto em cada ponto.")


### O coeficiente R² — o modelo é bom?

Depois de ajustar a reta, precisamos de uma forma de medir **o quão bem ela
representa os dados**. Para isso usamos o **R² (coeficiente de determinação)**.

A ideia é simples: comparamos o erro do nosso modelo com o erro que teríamos se
apenas **chutássemos sempre a média**. Quanto melhor o modelo em relação a esse
"chute ingênuo", mais perto de 1 fica o R².

| R² | Interpretação |
|----|---------------|
| **1.00** | Modelo perfeito |
| **0.70** | O modelo explica 70% da variação de Y |
| **0.00** | Tanto faz usar o modelo ou simplesmente prever a média |
| **< 0** | O modelo é pior do que só usar a média! |

<div style="background:#fff3cd; border-left:5px solid #856404; padding:12px 18px; border-radius:6px; margin:12px 0;">
<strong style="color:#856404;">Atenção:</strong>
<span style="color:#856404;"> R² alto não garante um bom modelo (pode ser overfitting), e R² baixo não
significa que o modelo é inútil — depende do problema.</span>
</div>


In [ ]:
# Calculando o R² da reta acima
from sklearn.metrics import r2_score

r2 = r2_score(y, y_hat)

print(f"R² = {r2:.4f}")
print(f"Isso significa que a reta explica {r2:.0%} da variação na tarifa média por classe.")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 2 — Repita o mesmo processo, mas agora usando o <strong>tamanho da família</strong> como x (em vez da classe) para prever a tarifa média. Preencha o código abaixo e responda: o R² ficou maior ou menor que o da classe? O que isso sugere sobre qual variável é mais importante para prever a tarifa?</span></div>

In [ ]:
# ✏️ MISSÃO 2 — complete o código

# Dica: use df.groupby("tamanho_familia")["fare"].mean() para agrupar
tamanhos = sorted(df["tamanho_familia"].unique())
x_fam = np.array(tamanhos)
y_fam = np.array([df[df.tamanho_familia == t]["fare"].mean() for t in tamanhos])

# ✏️ Ajuste a reta com np.polyfit (assim como fizemos para 'classe')
# beta1_fam, beta0_fam = ???

# ✏️ Calcule as previsões e o R²
# y_hat_fam = ???
# r2_fam = r2_score(y_fam, y_hat_fam)

# print(f"R² (tamanho da família): {r2_fam:.4f}")
# print(f"R² (classe):             {r2:.4f}")


In [ ]:
# ── GABARITO DA MISSÃO 2 (descomente para ver) ───────────────────────────────
# beta1_fam, beta0_fam = np.polyfit(x_fam, y_fam, 1)
# y_hat_fam = beta0_fam + beta1_fam * x_fam
# r2_fam = r2_score(y_fam, y_hat_fam)
#
# print(f"Reta: ŷ = {beta0_fam:.2f} + ({beta1_fam:.2f}) · x")
# print(f"R² (tamanho da família): {r2_fam:.4f}")
# print(f"R² (classe):             {r2:.4f}")
# print()
# print("Quem tiver o maior R² é a variável que, sozinha, melhor explica a tarifa.")


*✏️ O R² do tamanho da família foi `???` que o da classe.*

*✏️ Isso sugere que `???` é a variável mais importante (sozinha) para prever a tarifa.*


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 3</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Regressão Linear Múltipla no Titanic</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Uma feature raramente é suficiente — combinamos várias para prever melhor."</p>
    </div>
</div>

Até agora usamos **uma única feature** por vez (classe, depois família). Mas o
preço da passagem depende de várias coisas ao mesmo tempo. A **Regressão
Múltipla** combina todas elas em uma única equação:

```
ŷ = β₀ + β₁·x₁ + β₂·x₂ + ... + βₙ·xₙ
```

Cada `β` mostra o quanto aquela feature "empurra" a tarifa para cima ou para
baixo, **mantendo as outras features fixas**. Vamos treinar esse modelo usando
`scikit-learn`, que faz todo o trabalho pesado por nós.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Features escolhidas para prever a tarifa
FEATURES = ["pclass", "sex_enc", "age", "tamanho_familia"]

X = df[FEATURES]
y = df["fare"]

# Separando treino (80%) e teste (20%) — o modelo nunca vê os dados de teste
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = LinearRegression()
modelo.fit(X_tr, y_tr)

y_pred = modelo.predict(X_te)

print("✅ Regressão Linear Múltipla treinada!")
print(f"   Intercepto (β₀): £{modelo.intercept_:.2f}")
print(f"   R² no teste:     {r2_score(y_te, y_pred):.4f}")


In [ ]:
# Quais features mais influenciam a tarifa prevista?
coef_df = pd.DataFrame({
    "feature":     FEATURES,
    "coeficiente": modelo.coef_
}).sort_values("coeficiente", key=abs, ascending=False)

plt.figure(figsize=(8, 4))
cores = ["#0f3460" if v > 0 else "#e94560" for v in coef_df["coeficiente"]]
plt.barh(coef_df["feature"], coef_df["coeficiente"], color=cores, edgecolor="white")
plt.axvline(0, color="black", linewidth=1.2)
plt.xlabel("Coeficiente β")
plt.title("Impacto de cada feature na tarifa prevista\n(Azul = aumenta | Vermelho = diminui)",
          fontweight="bold")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(coef_df.to_string(index=False))


In [ ]:
# Previsto vs Real — quanto mais perto da linha diagonal, melhor o modelo
plt.figure(figsize=(6, 6))
max_fare = 300
plt.scatter(y_te, y_pred, alpha=0.4, color="#0f3460", s=25, edgecolors="none")
plt.plot([0, max_fare], [0, max_fare], color="#e94560", linestyle="--",
         linewidth=2, label="Previsão perfeita")
plt.xlabel("Tarifa Real (£)")
plt.ylabel("Tarifa Prevista (£)")
plt.title(f"Previsto vs Real — R² = {r2_score(y_te, y_pred):.3f}", fontweight="bold")
plt.xlim(0, max_fare); plt.ylim(0, max_fare)
plt.legend()
plt.tight_layout()
plt.show()

print("Pontos sobre a linha vermelha = previsão perfeita.")
print("Pontos acima da linha = o modelo superestimou a tarifa.")
print("Pontos abaixo da linha = o modelo subestimou a tarifa.")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 3 — Observe o gráfico de coeficientes e responda: (a) qual feature tem o maior impacto (positivo ou negativo) na tarifa? Faz sentido com o que você sabe do Titanic? (b) no gráfico Previsto vs Real, o modelo erra mais para tarifas baixas ou altas?</span></div>

*✏️ (a) Feature de maior impacto: `???` — faz sentido porque: `???`*

*✏️ (b) O modelo erra mais para tarifas `???`*


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 4</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Avaliando o Modelo: Métricas e Resíduos</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Um número só de R² não conta toda a história."</p>
    </div>
</div>

### Três métricas simples

| Métrica | O que mede | Como interpretar |
|---------|-----------|-------------------|
| **MAE** | Erro médio absoluto | "Em média, erramos £X para mais ou para menos" |
| **RMSE** | Erro médio, mas penalizando mais os erros grandes | Parecido com o MAE, mas mais sensível a outliers |
| **R²** | Fração da variação de Y explicada pelo modelo | 0 a 1 — quanto mais perto de 1, melhor |

**Dica prática:** se o RMSE for bem maior que o MAE, é sinal de que existem alguns
casos com erro muito grande (outliers) puxando o RMSE para cima.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae  = mean_absolute_error(y_te, y_pred)
rmse = np.sqrt(mean_squared_error(y_te, y_pred))
r2   = r2_score(y_te, y_pred)

print("Métricas do modelo no conjunto de teste:")
print(f"  MAE  = £{mae:.2f}   (em média, erramos essa quantia)")
print(f"  RMSE = £{rmse:.2f}")
print(f"  R²   = {r2:.4f}   ({r2:.0%} da variação explicada)")
print()
print(f"Razão RMSE/MAE = {rmse/mae:.2f}x")
print("  " + ("⚠️  Razão alta — provavelmente há outliers puxando o erro para cima"
              if rmse/mae > 2 else "✅ Razão normal — erros bem distribuídos"))


### Olhando para os resíduos

O **resíduo** de cada previsão é simplesmente `erro = valor real − valor previsto`.
Olhar para os resíduos ajuda a identificar **onde** o modelo está errando mais —
e por quê.

Um bom sinal é quando os resíduos aparecem **espalhados aleatoriamente em torno
de zero**, sem nenhum padrão. Se aparecer um "funil" (o erro cresce conforme a
tarifa aumenta), é sinal de que o modelo tem mais dificuldade nas tarifas altas.


In [ ]:
# Gráfico de resíduos: erro (real - previsto) vs valor previsto
residuos = y_te.values - y_pred

plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuos, alpha=0.4, color="#0f3460", s=25, edgecolors="none")
plt.axhline(0, color="#e94560", linestyle="--", linewidth=2)
plt.xlabel("Valor Previsto (£)")
plt.ylabel("Resíduo (Real − Previsto)")
plt.title("Resíduos do Modelo — ideal: pontos espalhados aleatoriamente em torno de zero",
          fontweight="bold")
plt.tight_layout()
plt.show()


<div style="background:#e8d5f5; border-left:5px solid #5b2c8d; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#5b2c8d;">💡 Por curiosidade — </strong><span style="color:#5b2c8d;">Se você notou um "funil" no gráfico acima (erro maior nas tarifas altas), uma solução comum é prever <code>log(fare)</code> em vez de <code>fare</code> diretamente, e depois desfazer o log na previsão final. Isso "comprime" os valores altos e ajuda o modelo a errar de forma mais proporcional. Não é obrigatório saber fazer isso hoje — é só para você saber que essa ferramenta existe.</span></div>

<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 4 (final) — Com tudo que vimos hoje, responda: (a) por que usamos treino e teste separados em vez de avaliar o modelo nos mesmos dados usados para treinar? (b) em que situação você preferirira usar MAE em vez de RMSE?</span></div>

*✏️ (a) Treino e teste separados servem para: `???`*

*✏️ (b) Eu usaria MAE em vez de RMSE quando: `???`*


**✏️ Minha reflexão sobre a aula:**

1. A principal diferença entre classificação e regressão é: *...*

2. O conceito que fez mais sentido para mim hoje foi: *...*

3. O conceito que eu preciso revisar é: *...*
